# Step 2: Base Model Training
Here we train our base learners: XGBoost, Random Forest, and Logistic Regression. XGBoost acts as our primary strong learner. We save their output probabilities for the Fuzzy Ensemble.

In [2]:
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Load data
X_train, y_train, X_test, y_test = joblib.load('processed_data.pkl')

print("Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Extract Probabilities (Probability of Class 1: Diabetic)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
rf_probs = rf_model.predict_proba(X_test)[:, 1]
lr_probs = lr_model.predict_proba(X_test)[:, 1]

# Save probabilities and true labels to disk
joblib.dump({'xgb': xgb_probs, 'rf': rf_probs, 'lr': lr_probs, 'y_test': y_test}, 'model_predictions.pkl')

print("\n--- Base Model Evaluation ---")
for name, probs in [('XGBoost', xgb_probs), ('Random Forest', rf_probs), ('Logistic Regression', lr_probs)]:
    preds = (probs > 0.5).astype(int)
    print(f"\n[{name}]")
    print(classification_report(y_test, preds))
    print(f"ROC AUC: {roc_auc_score(y_test, probs):.4f}")

Training XGBoost...


/Users/gagansinghal/Desktop/diabetes_prediction_project/.venv/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [20:01:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Random Forest...
Training Logistic Regression...

--- Base Model Evaluation ---

[XGBoost]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     17534
           1       0.91      0.71      0.80      1696

    accuracy                           0.97     19230
   macro avg       0.94      0.85      0.89     19230
weighted avg       0.97      0.97      0.97     19230

ROC AUC: 0.9741

[Random Forest]
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     17534
           1       0.78      0.74      0.76      1696

    accuracy                           0.96     19230
   macro avg       0.88      0.86      0.87     19230
weighted avg       0.96      0.96      0.96     19230

ROC AUC: 0.9638

[Logistic Regression]
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     17534
           1       0.42      0.87      0.57      1696

    accura